# Exercise 2.4 — Unfolding a Many-Body Spectrum

**Chapter 2: Random Matrix Theory and Spectral Statistics** &nbsp;|&nbsp; *Exercises*

---

Unfolding is the one preprocessing step that every level-statistics diagnostic depends on,
and it is the step newcomers most often get wrong, usually without noticing. This notebook
works it out end to end for a real many-body Hamiltonian, explaining the mathematics and the
numerics at every step, and then shows the two ways the procedure can silently fail.

We use the **mixed-field Ising chain**, the standard chaotic spin model,
$$
\hat H \;=\; J\sum_i \hat Z_i\hat Z_{i+1} \;+\; h_x\sum_i \hat X_i \;+\; h_z\sum_i \hat Z_i,
\qquad J=1,\; h_x=1.05,\; h_z=0.5,
$$
with open boundary conditions. A transverse field alone ($h_z=0$) is integrable through the
Jordan--Wigner transformation; the longitudinal field $h_z$ breaks that integrability and makes
the chain chaotic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- build the mixed-field Ising Hamiltonian by dense Kronecker products ---
SX = np.array([[0, 1], [1, 0]], dtype=float)
SZ = np.diag([1.0, -1.0])
I2 = np.eye(2)

def op(single, site, n):
    "single-site operator `single` acting on `site` of an n-qubit chain"
    m = np.array([[1.0]])
    for k in range(n):
        m = np.kron(m, single if k == site else I2)
    return m

def mfi_hamiltonian(n, J=1.0, hx=1.05, hz=0.5):
    H = np.zeros((2**n, 2**n))
    for i in range(n - 1):
        H += J * op(SZ, i, n) @ op(SZ, i + 1, n)
    for i in range(n):
        H += hx * op(SX, i, n) + hz * op(SZ, i, n)
    return H

N = 12
H = mfi_hamiltonian(N)
E_full = np.sort(np.linalg.eigvalsh(H))
print(f"N = {N}, Hilbert dimension D = {2**N}, {len(E_full)} eigenvalues")

## Step 0 — The obstacle: raw spacings are not comparable

The nearest-neighbour spacings $s_n = E_{n+1}-E_n$ of a many-body spectrum are dominated by the
**density of states**, which is far from uniform. A many-body band is roughly Gaussian: levels
pile up exponentially near the centre and thin out toward the edges. A spacing of a given size
means something completely different in the dense middle than at the sparse edge, so a raw
histogram of $s_n$ measures the global shape of the spectrum, not the local correlations we care
about. This is what unfolding removes.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
# (left) the density of states: strongly peaked, not uniform
a1.hist(E_full, bins=80, color="#4361ee", alpha=0.8)
a1.set_xlabel("energy $E$"); a1.set_ylabel("count")
a1.set_title("(a) density of states: Gaussian, not uniform")
# (right) raw spacings: shaped by the density, no universal form
s_raw = np.diff(E_full)
a2.hist(s_raw / s_raw.mean(), bins=80, density=True, color="#d62828", alpha=0.8)
a2.set_xlabel("$s / \\langle s\\rangle$"); a2.set_ylabel("$P(s)$")
a2.set_title("(b) raw spacings: dominated by the density")
plt.tight_layout(); plt.show()
print("The raw spacing histogram reflects the varying density, not level repulsion.")

## Step 1 — Symmetry first (the trap before unfolding)

Before unfolding, resolve the symmetries. The chain has a **reflection symmetry** $i\mapsto N-1-i$.
Its Hamiltonian is block-diagonal in the two reflection-parity sectors, and eigenvalues from
different blocks are statistically **independent**. Level statistics on the full spectrum therefore
superpose two uncorrelated sequences, which dilutes the level repulsion (an eigenvalue from one
sector feels no repulsion from those of the other) and pulls the gap ratio
$$
\langle r\rangle = \Big\langle \frac{\min(s_n, s_{n+1})}{\max(s_n, s_{n+1})}\Big\rangle
$$
down from the Gaussian-orthogonal value $\approx 0.53$ toward the Poisson value $0.386$.

The gap ratio is used here because it needs **no** unfolding: being a ratio of adjacent spacings,
it is insensitive to the local density. It is the right tool for checking chaos; it is *not* a
substitute for unfolding when you want the full spacing distribution or the form factor.

In [ ]:
def gap_ratio(E):
    E = np.sort(E); m = len(E)
    E = E[m // 5 : 4 * m // 5]          # central 60%, away from the edges
    s = np.diff(E); s = s[s > 1e-9]
    r = np.minimum(s[:-1], s[1:]) / np.maximum(s[:-1], s[1:])
    return float(np.mean(r))

def reflection_operator(n):
    "permutation matrix reversing the bit order of each computational basis state"
    D = 2**n; P = np.zeros((D, D))
    for b in range(D):
        rb = 0
        for k in range(n):
            rb |= ((b >> k) & 1) << (n - 1 - k)
        P[rb, b] = 1.0
    return P

P = reflection_operator(N)
wP, vP = np.linalg.eigh(P)
even = vP[:, wP > 0]                      # +1 reflection-parity subspace
E_even = np.sort(np.linalg.eigvalsh(even.T @ H @ even))

print(f"full spectrum   <r> = {gap_ratio(E_full):.3f}   (two sectors mixed, looks sub-chaotic)")
print(f"parity-even     <r> = {gap_ratio(E_even):.3f}   (single sector, Gaussian-orthogonal ~0.53)")
print("\\nMixing two independent sequences hid the chaos. Always resolve symmetries first.")

## Step 2 — The mathematics of unfolding

Work inside one sector from now on. Define the **integrated density of states** (the staircase)
$$
N(E) = \#\{n : E_n \le E\},
$$
which jumps by one at each eigenvalue. Split it into a smooth average part and a fluctuating part,
$$
N(E) = \bar N(E) + N_{\mathrm{fluc}}(E).
$$
The smooth part $\bar N(E)$ is the system-specific global density, the thing we want to remove.
The **unfolded energies** are its values at the eigenvalues,
$$
\epsilon_n = \bar N(E_n).
$$
By construction $\epsilon_n$ counts, smoothly, how many levels lie below $E_n$, so consecutive
unfolded energies differ **on average by one**: the unfolded spectrum has unit mean spacing
everywhere, and only the universal fluctuations $N_{\mathrm{fluc}}$ survive in $s_n = \epsilon_{n+1}-\epsilon_n$.

Numerically, $\bar N(E)$ is obtained by fitting a low-order polynomial to the staircase over the
central part of the band. The polynomial degree is the one real choice, and Step 4 shows what
happens when it is wrong.

In [ ]:
# build the staircase N(E) for the resolved sector
m = len(E_even)
lo, hi = int(0.1 * m), int(0.9 * m)      # trim the edges: the density is least reliable there
Ec = E_even[lo:hi]
stair = np.arange(len(Ec))               # N(E_n) = n for sorted levels

# fit the smooth integrated density with a low-order polynomial
DEG = 9
coef = np.polyfit(Ec, stair, DEG)
Nbar = np.polyval(coef, Ec)              # smooth staircase evaluated at the eigenvalues

plt.figure(figsize=(6.5, 4))
plt.plot(Ec, Nbar, "-", color="#d62828", lw=2.2,
         label=fr"smooth $\bar N(E)$, degree {DEG}", zorder=2)
plt.plot(Ec[::6], stair[::6], "o", ms=4.5, mfc="none", mec="#4361ee", mew=1.1,
         label="staircase $N(E)$", zorder=3)   # open circles so the fit shows through
plt.xlabel("energy $E$"); plt.ylabel("integrated density")
plt.legend(); plt.title("fitting the smooth part of the staircase"); plt.tight_layout(); plt.show()

In [ ]:
# Step 3 — unfold and verify unit mean spacing
eps = np.polyval(coef, Ec)               # epsilon_n = Nbar(E_n)
s_unf = np.diff(eps)
print(f"mean unfolded spacing = {s_unf.mean():.4f}   (must be 1 by construction)")
print(f"variance of unfolded spacings = {s_unf.var():.4f}")
print(f"Wigner (GOE) prediction        = {4/np.pi - 1:.4f}")

## Step 3 — The payoff: the Wigner surmise appears

With the density removed, the unfolded spacing distribution collapses onto the **Wigner surmise**
for the Gaussian orthogonal ensemble,
$$
P(s) = \frac{\pi}{2}\, s\, e^{-\pi s^2/4},
$$
the hallmark of level repulsion: $P(s)\to 0$ as $s\to 0$, so nearby levels avoid each other. The raw
distribution of Step 0 showed nothing of the kind. This is the entire purpose of unfolding.

In [ ]:
def wigner(s):
    return (np.pi / 2) * s * np.exp(-np.pi * s**2 / 4)

s_norm = s_unf / s_unf.mean()
xs = np.linspace(0, 3.5, 200)
plt.figure(figsize=(6.5, 4))
plt.hist(s_norm, bins=60, density=True, color="#2a9d5c", alpha=0.7, label="unfolded chain")
plt.plot(xs, wigner(xs), "k--", lw=2, label=r"Wigner surmise $\frac{\pi}{2}s\,e^{-\pi s^2/4}$")
plt.xlabel("$s$"); plt.ylabel("$P(s)$"); plt.legend()
plt.title("unfolded spacing distribution vs Wigner"); plt.tight_layout(); plt.show()

## Step 4 — The two ways unfolding fails

The fit is where unfolding goes wrong, and it fails in two opposite ways.

**Under-fitting (too rigid).** A polynomial of degree $2$ cannot follow the Gaussian bulk of
$\bar N(E)$. Part of the global density is left behind and reappears in the spacings, inflating
their variance above the Wigner value $0.273$.

**Over-fitting (too flexible).** The opposite mistake is a fit so flexible that it follows the
staircase level by level. Here the polynomial degree is the wrong knob to worry about: a *global*
polynomial is rigid, and even degree $300$ on these $\sim\!1700$ levels stays close to Wigner,
because it simply cannot bend to reach every individual level. The real over-fit is a **local**
fit, a cubic spline with the smoothing turned down, which does chase the staircase; it forces
$\epsilon_{n+1}-\epsilon_n\approx 1$ everywhere and drives the variance to nearly zero, erasing the
repulsion the diagnostic exists to measure.

In [ ]:
from scipy.interpolate import UnivariateSpline
import warnings; warnings.simplefilter("ignore")

def var_of_unfolded(fitter):
    e = fitter(Ec); s = np.diff(e); return (s / s.mean()).var()

wig = 4/np.pi - 1
print(f"Wigner (GOE) variance target: {wig:.3f}\\n")
print("under-fitting / good fit (global polynomial):")
for deg in [2, 5, 9, 50, 300]:
    c = np.polyfit(Ec, stair, deg)
    v = var_of_unfolded(lambda E, c=c: np.polyval(c, E))
    tag = "  <- too rigid, residual density" if deg == 2 else ("  <- good" if deg in (5,9) else "  <- still fine: a global poly cannot over-fit")
    print(f"  degree {deg:3d}: Var(s) = {v:.3f}{tag}")

print("\\nover-fitting (local cubic spline, smoothing s decreasing):")
for smooth in [1e5, 1e2, 1.0]:
    sp = UnivariateSpline(Ec, stair, s=smooth, k=3)
    v = var_of_unfolded(sp)
    tag = "  <- over-fit: repulsion erased" if smooth == 1.0 else ""
    print(f"  spline s = {smooth:6.0e}: Var(s) = {v:.3f}{tag}")

In [ ]:
# visualize the two failures next to the good fit
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))   # independent y-axes: the over-fit spike must not squash the others
def unfold_poly(deg):
    c = np.polyfit(Ec, stair, deg); s = np.diff(np.polyval(c, Ec)); return s / s.mean()
def unfold_spline(smooth):
    sp = UnivariateSpline(Ec, stair, s=smooth, k=3); s = np.diff(sp(Ec)); return s / s.mean()

for ax, (s, title) in zip(axes, [
        (unfold_poly(2),   "(a) under-fit: degree 2\\n(residual density)"),
        (unfold_poly(9),   "(b) good: degree 9\\n(matches Wigner)"),
        (unfold_spline(1.0), "(c) over-fit: spline $s=1$\\n(repulsion erased)")]):
    ax.hist(s, bins=50, density=True, color="#888", alpha=0.7)
    ax.plot(xs, wigner(xs), "k--", lw=1.8)
    ax.set_xlabel("$s$"); ax.set_title(title); ax.set_xlim(0, 3.5)
    ax.set_ylabel("$P(s)$")
plt.tight_layout(); plt.show()

## Summary — the recipe

1. **Resolve every symmetry** and work inside one sector. Mixing independent sequences fakes
   Poisson statistics and hides the chaos (here it pulled $\langle r\rangle$ from $0.55$ to $0.42$).
2. **Build the staircase** $N(E)$ and **fit a low-order global polynomial** (degree a few to ten)
   to its central part, away from the unreliable edges.
3. **Unfold**: $\epsilon_n = \bar N(E_n)$. Check that the mean spacing is $1$.
4. **Only now** compute $P(s)$, the number variance, or the form factor.

The one judgement call is the fit. Guard against **under-fitting**, which leaves density behind;
a low-order global polynomial almost never over-fits, so the over-fitting one hears warned about is
really a hazard of flexible *local* fits (splines, kernel smoothers) rather than of polynomial
degree. When in doubt, use a modest global polynomial and confirm the unfolded variance sits near
the Wigner value $0.273$.